# Lesson 10.1 · The life of a project

In modules 1-9 you learned **tools**. In a real project most of the mistakes are not in the tools but in the **order** in which you use them and in what you take for granted. This lesson is a checklist with a runnable example; the three projects of this module put it to the test.

**What you will learn**

- The nine steps of a project, and at which step each typical mistake of the previous modules is born.
- How to build a correct flow in a few cells: closed test set, baseline, pipeline, comparison, choice and final check.
- Why the comparison with a "dumb" baseline is the first sanity check.

## The nine steps

| # | Step | Question | Typical mistake (where you saw it) |
|---|---|---|---|
| 1 | **Problem and metric** | What does a mistake cost? What is the prevalence? Which metric do I decide with? | Accuracy with imbalanced classes (M4) |
| 2 | **Split and closed test set** | The test set is touched **once**, at the end. Does the split respect the structure of the data (time, groups, areas)? | Random split on grouped or spatial data (M3, M10) |
| 3 | **Baseline** | What does the dumbest model do (`DummyClassifier`, the mean, a linear model)? | Not having a reference point (M1, M9 F2) |
| 4 | **Pipeline** | Imputation, scaling, encoding **inside** the pipeline, fitted only on the training part of each fold. | Leakage from preprocessing on the whole dataset (M2, M3) |
| 5 | **Comparison** | Same folds, same metric, with the variability (mean and deviation). | Comparison with a single seed or a single partition (M8 E1) |
| 6 | **Tuning** | Only on the training set, with nested validation or a separate validation set. | Selection optimism (M3, M9 B3) |
| 7 | **Final check** | A single shot on the test set, with an uncertainty interval. | Choosing by looking at the test (M9 E1) |
| 8 | **Decision** | Threshold and costs, calibration, prediction intervals. | Threshold 0.5 out of habit (M4 B2) |
| 9 | **Explanation and report** | Importances on unseen data, single cases, limits and risks. | Importance computed on the training set (M8 E3) |

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans(n=2000, seed=4)
y = df.pop("defaulted")
X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.25, stratify=y, random_state=0)      # step 2: the test set is closed here
print(f"training {len(X_tr)} rows, test {len(X_te)} | prevalence of defaulters: {y_tr.mean():.1%}")

## Steps 3-5: baseline, pipeline, comparison

The `DummyClassifier` that always predicts the most frequent class has an accuracy equal to the prevalence (about 80%) and an AUC of 0.5: **it is the score to beat**. Then a linear pipeline and a tree-based one, on the **same folds**.

In [ ]:
numeric = ["age", "income", "job_tenure", "amount"]
nominal = ["city", "contract_type", "occupation"]
order = [ds.EDUCATION_ORDER] if hasattr(ds, "EDUCATION_ORDER") else None


def preprocessor():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric),
        ("edu", make_pipeline(OrdinalEncoder(categories=order, handle_unknown="use_encoded_value", unknown_value=-1),
                              SimpleImputer(strategy="median"), StandardScaler()), ["education"]),
        ("nom", OneHotEncoder(handle_unknown="ignore", min_frequency=15), nominal),
    ])


cv = StratifiedKFold(5, shuffle=True, random_state=0)
candidates = {
    "baseline (most frequent class)": make_pipeline(preprocessor(), DummyClassifier(strategy="prior")),
    "logistic regression": make_pipeline(preprocessor(), LogisticRegression(max_iter=2000)),
}
rows = {}
for name, model in candidates.items():
    r = cross_validate(model, X_tr, y_tr, cv=cv, scoring=["roc_auc", "accuracy"])
    rows[name] = {"mean AUC": r["test_roc_auc"].mean(), "AUC std": r["test_roc_auc"].std(), "accuracy": r["test_accuracy"].mean()}
pd.DataFrame(rows).T.round(3)

The accuracy of the model (about 0.85) is only slightly above that of the baseline (0.80): looking only at that you would say the model "is not needed". The **AUC** (0.82 against 0.50) says the opposite: the model ranks people well by risk, even if at threshold 0.5 it does not change
many decisions. With imbalanced classes the right metric is almost always probabilistic or a ranking one, then the threshold is chosen with the costs (step 8).

## Steps 7-8: final check and decision

Only after choosing **one** model do you open the test set, once. The threshold is chosen with the **out-of-sample predictions on the training set** (`cross_val_predict`), not on the test.

In [ ]:
from sklearn.model_selection import cross_val_predict

model = candidates["logistic regression"]
p_cv = cross_val_predict(model, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
cost = lambda y_true, p, threshold: int(((p >= threshold) & (y_true == 0)).sum() * 1 + ((p < threshold) & (y_true == 1)).sum() * 8)      # a missed defaulter costs 8 times a wrong refusal
thresholds = np.linspace(0.02, 0.9, 89)
threshold = thresholds[int(np.argmin([cost(y_tr.to_numpy(), p_cv, s) for s in thresholds]))]

p_te = model.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
print(f"AUC on the test (only once): {roc_auc_score(y_te, p_te):.3f} | threshold chosen on the training set: {threshold:.2f}")
print(f"cost on the test: threshold 0.5 -> {cost(y_te.to_numpy(), p_te, 0.5)} | threshold {threshold:.2f} -> {cost(y_te.to_numpy(), p_te, threshold)} | refuse everyone -> {int((y_te == 0).sum())}")

> **Common mistake.** Choosing the threshold (or the model, or the hyperparameters) by looking at the test set "just to see". From that moment the test set has become a validation set, and the number you report is optimistic without any warning stopping you.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why must the baseline be computed with the same folds and the same metric as the model?</summary>

Because it serves to answer the question "how much is the model worth compared with doing nothing intelligent?": with different folds or metrics the difference also contains the difference in protocol. With imbalanced classes the baseline has high accuracy and AUC 0.5: it immediately shows which metric is informative.

</details>

<details><summary>Why is the threshold chosen on the out-of-sample predictions of the training set and not on the test?</summary>

Because the test set must simulate data never used for any decision. The threshold is a hyperparameter: chosen on the test, the reported cost is optimistic. With `cross_val_predict` every prediction comes from a model that has not seen that example, so it is an honest estimate to choose with.

</details>